In [1]:
# =========================
# STEP 3 — K-SHAPE CLUSTERING (single cell, independent, NO PANDAS)
# Input priority:
#   1) /user/data/clustering_input/train_series_long
#   2) fallback: /user/data/preprocess/model_ready (train split only)
# Output:
#   - /workspace/code/kshape/clustering_meta/cluster_assignments_final.csv
#   - /user/data/clustering/panel_with_cluster_id
#   - /workspace/code/kshape/clustering_meta/kshape_metadata.json
# =========================

import os
import json
import csv
import warnings
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

# -------------------------
# 1) CONFIG
# -------------------------
CLUSTER_INPUT_PATH = "/user/data/clustering_input/train_series_long"
FALLBACK_MODEL_READY_PATH = "/user/data/preprocess/model_ready"
FULL_PANEL_PATH = "/user/data/preprocess/full_panel"

ASSIGNMENTS_CSV_OUT = "/workspace/code/kshape/clustering_meta/cluster_assignments_final.csv"
CLUSTER_META_DIR = "/workspace/code/kshape/clustering_meta"
PANEL_WITH_CLUSTER_OUT = "/user/data/clustering/panel_with_cluster_id"

N_CLUSTERS = 3
RANDOM_STATE = 42
AUTO_INSTALL_TSLEARN = False

# Guard để tránh collect ma trận quá lớn về driver
MAX_MATRIX_CELLS = 20_000_000

os.makedirs(CLUSTER_META_DIR, exist_ok=True)

spark = SparkSession.builder.appName("KShapeClustering").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.parquet.mergeSchema", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.sparkContext.setLogLevel("WARN")

# -------------------------
# 2) HELPERS
# -------------------------
def hdfs_path_exists(path_str: str) -> bool:
    jvm = spark._jvm
    hconf = spark._jsc.hadoopConfiguration()
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(hconf)
    Path = jvm.org.apache.hadoop.fs.Path
    return fs.exists(Path(path_str))

def load_tslearn_kshape():
    try:
        from tslearn.clustering import KShape
        return KShape
    except Exception as e:
        if AUTO_INSTALL_TSLEARN:
            import sys, subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "tslearn"])
            from tslearn.clustering import KShape
            return KShape
        raise ImportError(
            "Không import được tslearn.KShape. "
            "Hãy cài trước bằng lệnh: pip install tslearn "
            "hoặc bật AUTO_INSTALL_TSLEARN=True nếu môi trường cho phép."
        ) from e

def z_norm_1d(x):
    x = np.asarray(x, dtype=np.float64)
    m = np.nanmean(x)
    s = np.nanstd(x)
    if np.isnan(s) or s < 1e-12:
        s = 1.0
    return (x - m) / s

# -------------------------
# 3) READ CLUSTER INPUT
# -------------------------
if hdfs_path_exists(CLUSTER_INPUT_PATH):
    cluster_long = spark.read.parquet(CLUSTER_INPUT_PATH)
    cluster_input_source = CLUSTER_INPUT_PATH
else:
    if not hdfs_path_exists(FALLBACK_MODEL_READY_PATH):
        raise ValueError(
            f"Không tìm thấy cả CLUSTER_INPUT_PATH ({CLUSTER_INPUT_PATH}) "
            f"và FALLBACK_MODEL_READY_PATH ({FALLBACK_MODEL_READY_PATH})"
        )

    fallback = spark.read.parquet(FALLBACK_MODEL_READY_PATH)
    split_col = "dataset_split" if "dataset_split" in fallback.columns else ("split" if "split" in fallback.columns else None)
    if split_col is not None:
        fallback = fallback.filter(F.col(split_col) == "train")

    cluster_long = fallback.select(
        F.col("pickup_bin_30m").cast("timestamp").alias("pickup_bin_30m"),
        F.col("PULocationID").cast("int").alias("PULocationID"),
        F.coalesce(F.col("pickup_demand"), F.lit(0.0)).cast("double").alias("pickup_demand")
    )
    cluster_input_source = FALLBACK_MODEL_READY_PATH

required_cols = {"pickup_bin_30m", "PULocationID", "pickup_demand"}
missing = required_cols - set(cluster_long.columns)
if missing:
    raise ValueError(f"Thiếu cột bắt buộc cho clustering: {sorted(list(missing))}")

cluster_long = (
    cluster_long
    .filter(F.col("pickup_bin_30m").isNotNull())
    .filter(F.col("PULocationID").isNotNull())
    .withColumn("PULocationID", F.col("PULocationID").cast("int"))
    .withColumn("pickup_bin_30m", F.col("pickup_bin_30m").cast("timestamp"))
    .withColumn("pickup_demand", F.coalesce(F.col("pickup_demand"), F.lit(0.0)).cast("double"))
)

# -------------------------
# 4) CLEAN + AGG LONG FORM
#    Không pivot trong Spark
# -------------------------
cluster_long_agg = (
    cluster_long
    .groupBy("PULocationID", "pickup_bin_30m")
    .agg(F.sum("pickup_demand").alias("pickup_demand"))
)

n_loc = cluster_long_agg.select("PULocationID").distinct().count()
n_bin = cluster_long_agg.select("pickup_bin_30m").distinct().count()
n_cell_est = n_loc * n_bin

print(f"cluster_input_source = {cluster_input_source}")
print(f"n_locations          = {n_loc}")
print(f"n_time_bins          = {n_bin}")
print(f"estimated_matrix     = {n_cell_est:,} cells")

if n_loc == 0:
    raise ValueError("Không có location nào để clustering.")
if n_bin == 0:
    raise ValueError("Không có time bin nào để clustering.")
if n_loc < N_CLUSTERS:
    raise ValueError(f"Số location ({n_loc}) nhỏ hơn N_CLUSTERS={N_CLUSTERS}")
if n_cell_est > MAX_MATRIX_CELLS:
    raise MemoryError(
        f"Ma trận quá lớn để collect an toàn: {n_cell_est:,} cells > {MAX_MATRIX_CELLS:,}. "
        f"Hãy giảm time range hoặc downsample trước khi clustering."
    )

# -------------------------
# 5) BUILD GLOBAL TIME INDEX
# -------------------------
time_index_rows = (
    cluster_long_agg
    .select("pickup_bin_30m")
    .distinct()
    .orderBy("pickup_bin_30m")
    .collect()
)

time_index = [r["pickup_bin_30m"] for r in time_index_rows]
time_pos = {t: i for i, t in enumerate(time_index)}

if len(time_index) == 0:
    raise ValueError("time_index rỗng, không thể tạo ma trận K-Shape.")

# -------------------------
# 6) GROUP SERIES PER LOCATION
#    Lưu ý: collect_list sau shuffle không đảm bảo order,
#    nên dùng array_sort trên struct(time, value)
# -------------------------
series_rows = (
    cluster_long_agg
    .groupBy("PULocationID")
    .agg(
        F.array_sort(
            F.collect_list(
                F.struct(
                    F.col("pickup_bin_30m").alias("pickup_bin_30m"),
                    F.col("pickup_demand").alias("pickup_demand")
                )
            )
        ).alias("series")
    )
    .orderBy("PULocationID")
    .collect()
)

if len(series_rows) == 0:
    raise ValueError("Không có series_rows để dựng ma trận clustering.")

# -------------------------
# 7) BUILD NUMPY MATRIX DIRECTLY
#    Không dùng pandas
# -------------------------
loc_ids = []
X_list = []

for row in series_rows:
    loc_id = int(row["PULocationID"])
    arr = np.zeros(len(time_index), dtype=np.float64)

    for item in row["series"]:
        t = item["pickup_bin_30m"]
        v = float(item["pickup_demand"]) if item["pickup_demand"] is not None else 0.0
        idx = time_pos.get(t, None)
        if idx is not None:
            arr[idx] = v

    loc_ids.append(loc_id)
    X_list.append(arr)

if len(X_list) == 0:
    raise ValueError("Không tạo được ma trận X cho K-Shape.")

X = np.stack(X_list, axis=0)

print(f"X shape               = {X.shape}")

if X.shape[0] < N_CLUSTERS:
    raise ValueError(f"Số location ({X.shape[0]}) nhỏ hơn N_CLUSTERS={N_CLUSTERS}")

# z-normalize theo từng chuỗi
Xz = np.vstack([z_norm_1d(x) for x in X])

# tslearn KShape an toàn hơn với shape 3D: (n_samples, sz, d)
Xts = Xz[:, :, np.newaxis]

# -------------------------
# 8) K-SHAPE FIT
# -------------------------
KShape = load_tslearn_kshape()
model = KShape(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=1,
    verbose=False
)

cluster_labels = model.fit_predict(Xts)

# -------------------------
# 9) EXPORT ASSIGNMENTS CSV
# -------------------------
assignment_rows = [(int(loc_id), int(label)) for loc_id, label in zip(loc_ids, cluster_labels)]

with open(ASSIGNMENTS_CSV_OUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["PULocationID", "cluster_id"])
    writer.writerows(assignment_rows)

# optional centroids export
centroids = np.asarray(model.cluster_centers_).squeeze(-1)
np.save(os.path.join(CLUSTER_META_DIR, "kshape_centroids.npy"), centroids)

# -------------------------
# 10) JOIN BACK TO FULL PANEL
# -------------------------
if not hdfs_path_exists(FULL_PANEL_PATH):
    raise ValueError(f"Không tìm thấy FULL_PANEL_PATH: {FULL_PANEL_PATH}")

full_panel = spark.read.parquet(FULL_PANEL_PATH)

assignments_schema = T.StructType([
    T.StructField("PULocationID", T.IntegerType(), False),
    T.StructField("cluster_id", T.IntegerType(), False),
])

assignments_spark = spark.createDataFrame(assignment_rows, schema=assignments_schema)

panel_with_cluster = (
    full_panel
    .join(F.broadcast(assignments_spark), on="PULocationID", how="left")
    .withColumn("cluster_id", F.col("cluster_id").cast("int"))
)

partition_cols = []
if "dataset_split" in panel_with_cluster.columns:
    partition_cols.append("dataset_split")
if "month" in panel_with_cluster.columns:
    partition_cols.append("month")

writer = panel_with_cluster.write.mode("overwrite")
if partition_cols:
    writer = writer.partitionBy(*partition_cols)
writer.parquet(PANEL_WITH_CLUSTER_OUT)

# -------------------------
# 11) METADATA
# -------------------------
metadata = {
    "cluster_input_source": cluster_input_source,
    "full_panel_path": FULL_PANEL_PATH,
    "assignments_csv_out": ASSIGNMENTS_CSV_OUT,
    "panel_with_cluster_out": PANEL_WITH_CLUSTER_OUT,
    "n_clusters": int(N_CLUSTERS),
    "random_state": int(RANDOM_STATE),
    "n_locations": int(len(loc_ids)),
    "n_time_bins": int(len(time_index)),
    "matrix_shape": [int(X.shape[0]), int(X.shape[1])],
    "max_matrix_cells_guard": int(MAX_MATRIX_CELLS),
}

with open(os.path.join(CLUSTER_META_DIR, "kshape_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# -------------------------
# 12) PRINT SUMMARY
# -------------------------
print("=" * 120)
print("STEP 3 — K-SHAPE CLUSTERING DONE")
print("=" * 120)
for k, v in metadata.items():
    print(f"{k}: {v}")

print("\nCluster counts:")
cluster_counter = Counter(int(x) for x in cluster_labels)
for cid in sorted(cluster_counter):
    print(f"{cid}: {cluster_counter[cid]}")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-831fa014-3134-4564-b067-8e868fe947b5;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 113ms :: artifacts dl 4ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

cluster_input_source = /user/data/clustering_input/train_series_long
n_locations          = 66
n_time_bins          = 4744
estimated_matrix     = 313,104 cells
X shape               = (66, 4744)


26/03/31 04:21:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


STEP 3 — K-SHAPE CLUSTERING DONE
cluster_input_source: /user/data/clustering_input/train_series_long
full_panel_path: /user/data/preprocess/full_panel
assignments_csv_out: /workspace/code/kshape/clustering_meta/cluster_assignments_final.csv
panel_with_cluster_out: /user/data/clustering/panel_with_cluster_id
n_clusters: 3
random_state: 42
n_locations: 66
n_time_bins: 4744
matrix_shape: [66, 4744]
max_matrix_cells_guard: 20000000

Cluster counts:
0: 36
1: 23
2: 7
